In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# Load dataset
file_path = "/kaggle/input/data-science-income-prediction/train.csv"
df = pd.read_csv(file_path, low_memory=False)

# Drop irrelevant columns
drop_columns = ["id", "address_role", "zip_code", "deposit_balance", "deposit_incoming", "children", "credit_history_3_years", "credit_history_1_year", "housing_type"]
df.drop(columns=drop_columns, inplace=True, errors='ignore')

# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Encode categorical variables (using same encoders for train and test)
label_encoders = {}
for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = {k: v for v, k in enumerate(le.classes_)}  # Store encoder as dictionary

# Drop columns with too many missing values (threshold 50%)
df = df.dropna(thresh=len(df) * 0.5, axis=1)

# Fill remaining missing values
df.fillna(df.mean(), inplace=True)

# Select numeric features (excluding target column)
numeric_features = df.columns.tolist()
numeric_features.remove("primary_income")  # Target variable

# Use StandardScaler for both features and target
scaler = StandardScaler()
df[numeric_features] = scaler.fit_transform(df[numeric_features])

y_scaler = StandardScaler()
y = y_scaler.fit_transform(df["primary_income"].values.reshape(-1, 1))

# Check min and max values of y
print(f"y_train min: {y.min()}, max: {y.max()}, mean: {y.mean()}")

# Prepare training data
X = df[numeric_features].values

# Reshape input data to be 3D (samples, time steps, features)
X = np.reshape(X, (X.shape[0], 1, X.shape[1]))

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define LSTM model
model = Sequential([
    LSTM(512, return_sequences=True, input_shape=(1, X.shape[2])),
    BatchNormalization(),
    Dropout(0.2),
    LSTM(256, return_sequences=False),
    BatchNormalization(),
    Dropout(0.2),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(1)
])

# Compile model
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')

# Early Stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train model for 50 epochs
model.fit(X_train, y_train, epochs=50, batch_size=64, validation_data=(X_test, y_test), verbose=1, callbacks=[early_stopping])

# Predict and rescale
predictions = model.predict(X_test)
print(f"Predictions before inverse transform: min={np.min(predictions)}, max={np.max(predictions)}")
predictions = y_scaler.inverse_transform(predictions)
y_test = y_scaler.inverse_transform(y_test)

# Evaluate model
mse = np.mean((predictions - y_test) ** 2)
print(f"Mean Squared Error: {mse}")

# Load test dataset
test_file_path = "/kaggle/input/data-science-income-prediction/test.csv"
test_df = pd.read_csv(test_file_path, low_memory=False)

# Drop irrelevant columns
test_df.drop(columns=drop_columns, inplace=True, errors='ignore')

# Encode categorical variables in test set using trained encoders
for col in categorical_cols:
    test_df[col] = test_df[col].fillna("Unknown")
    if col in label_encoders:
        test_df[col] = test_df[col].astype(str).apply(lambda x: label_encoders[col].get(x, -1))  # Use -1 for unknown values
    else:
        test_df[col] = -1  # Default encoding for unknown values

# Ensure test data has the same preprocessing
test_df.fillna(df.mean(), inplace=True)
test_df[numeric_features] = scaler.transform(test_df[numeric_features])

# Check for NaN values after transformation
print("NaN count in test_df after scaling:", np.isnan(test_df[numeric_features]).sum().sum())

# Reshape test data to fit LSTM model
X_test_final = np.reshape(test_df.values, (test_df.shape[0], 1, test_df.shape[1]))

# Predict on test data
test_predictions = model.predict(X_test_final)
print(f"Predictions before inverse transform (test set): min={np.min(test_predictions)}, max={np.max(test_predictions)}")
test_predictions = y_scaler.inverse_transform(test_predictions)

# Fix any NaN predictions
test_predictions = np.nan_to_num(test_predictions, nan=0.0)

# Load sample submission file
submission_file_path = "/kaggle/input/data-science-income-prediction/sample_submission.csv"
submission_df = pd.read_csv(submission_file_path)

# Save predictions to the submission file
submission_df["primary_income"] = test_predictions

# Save the submission file
submission_output_path = "/kaggle/working/submission.csv"
submission_df.to_csv(submission_output_path, index=False)

print(f"Submission file saved at: {submission_output_path}")

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# Load dataset
file_path = "/kaggle/input/data-science-income-prediction/train.csv"
df = pd.read_csv(file_path, low_memory=False)

# Drop irrelevant columns
drop_columns = ["id", "address_role", "zip_code", "deposit_balance", "deposit_incoming", "children", "credit_history_3_years", "credit_history_1_year", "housing_type"]
df.drop(columns=drop_columns, inplace=True, errors='ignore')

# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Encode categorical variables (using same encoders for train and test)
label_encoders = {}
for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = {k: v for v, k in enumerate(le.classes_)}  # Store encoder as dictionary

# Drop columns with too many missing values (threshold 50%)
df = df.dropna(thresh=len(df) * 0.5, axis=1)

# Fill remaining missing values
df.fillna(df.mean(), inplace=True)

# Select numeric features (excluding target column)
numeric_features = df.columns.tolist()
numeric_features.remove("primary_income")  # Target variable

# Use StandardScaler for both features and target
scaler = StandardScaler()
df[numeric_features] = scaler.fit_transform(df[numeric_features])

y_scaler = StandardScaler()
y = y_scaler.fit_transform(df["primary_income"].values.reshape(-1, 1))

# Check min and max values of y
print(f"y_train min: {y.min()}, max: {y.max()}, mean: {y.mean()}")

# Prepare training data
X = df[numeric_features].values

# Reshape input data to be 3D (samples, time steps, features)
X = np.reshape(X, (X.shape[0], 1, X.shape[1]))

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define LSTM model
model = Sequential([
    LSTM(512, return_sequences=True, input_shape=(1, X.shape[2])),
    BatchNormalization(),
    Dropout(0.2),
    
    LSTM(256, return_sequences=True),
    BatchNormalization(),
    Dropout(0.2),
    
    LSTM(128, return_sequences=False),
    BatchNormalization(),
    Dropout(0.2),
    
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(1)
])

# Compile model with fixed learning rate
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')

# Early Stopping and Reduce Learning Rate on Plateau
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=0.00001)

# Train model for 50 epochs
model.fit(X_train, y_train, epochs=50, batch_size=64, validation_data=(X_test, y_test), verbose=1, callbacks=[early_stopping, reduce_lr])

# Predict and rescale
predictions = model.predict(X_test)
print(f"Predictions before inverse transform: min={np.min(predictions)}, max={np.max(predictions)}")
predictions = y_scaler.inverse_transform(predictions)
y_test = y_scaler.inverse_transform(y_test)

# Evaluate model
mse = np.mean((predictions - y_test) ** 2)
print(f"Mean Squared Error: {mse}")

# Load test dataset
test_file_path = "/kaggle/input/data-science-income-prediction/test.csv"
test_df = pd.read_csv(test_file_path, low_memory=False)

# Drop irrelevant columns
test_df.drop(columns=drop_columns, inplace=True, errors='ignore')

# Encode categorical variables in test set using trained encoders
for col in categorical_cols:
    test_df[col] = test_df[col].fillna("Unknown")
    if col in label_encoders:
        test_df[col] = test_df[col].astype(str).apply(lambda x: label_encoders[col].get(x, -1))  # Use -1 for unknown values
    else:
        test_df[col] = -1  # Default encoding for unknown values

# Ensure test data has the same preprocessing
test_df.fillna(df.mean(), inplace=True)
test_df[numeric_features] = scaler.transform(test_df[numeric_features])

# Check for NaN values after transformation
print("NaN count in test_df after scaling:", np.isnan(test_df[numeric_features]).sum().sum())

# Reshape test data to fit LSTM model
X_test_final = np.reshape(test_df.values, (test_df.shape[0], 1, test_df.shape[1]))

# Predict on test data
test_predictions = model.predict(X_test_final)
print(f"Predictions before inverse transform (test set): min={np.min(test_predictions)}, max={np.max(test_predictions)}")
test_predictions = y_scaler.inverse_transform(test_predictions)

# Fix any NaN predictions
test_predictions = np.nan_to_num(test_predictions, nan=0.0)

# Load sample submission file
submission_file_path = "/kaggle/input/data-science-income-prediction/sample_submission.csv"
submission_df = pd.read_csv(submission_file_path)

# Save predictions to the submission file
submission_df["primary_income"] = test_predictions

# Save the submission file
submission_output_path = "/kaggle/working/submission.csv"
submission_df.to_csv(submission_output_path, index=False)

print(f"Submission file saved at: {submission_output_path}")